# Klasifikasi Sampah Menggunakan CNN dan MobileNetV2

**Komputerisasi Industri Pertanian**

- **Nama:** Junervin  
- **NIM:** ................................  
- **Kelas:** ................................  
- **Dataset:** Garbage Classification (12 Classes) — Kaggle  
- **Metode:** Transfer Learning MobileNetV2

## 1. Strategi pemilihan kelas

Dataset asli memiliki 12 kelas, tetapi notebook ini menggunakan 6 kelas inti:

`battery`, `biological`, `cardboard`, `metal`, `paper`, dan `plastic`.

Alasannya:

- Keenam kelas memiliki jumlah gambar yang relatif seimbang.
- Total data sekitar 5.500 gambar sehingga training lebih ringan.
- Kelas tersebut relevan untuk pemilahan sampah.
- `clothes` dan `shoes` tidak digunakan karena sangat dominan dan sebagian gambar bukan sampah nyata.
- Tiga kelas kaca tidak digunakan karena mudah tertukar akibat warna dan pencahayaan.
- `trash` tidak digunakan karena isinya terlalu beragam.

Pemilihan ini bertujuan meningkatkan kestabilan model tanpa membuang makna utama proyek klasifikasi sampah.

## 2. Menyiapkan library

Cell berikut memasang KaggleHub, mengimpor library, menetapkan *random seed*, dan memeriksa apakah GPU tersedia.

In [ ]:
# Menyiapkan library dan konfigurasi agar hasil lebih konsisten.
!pip -q install kagglehub

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

## 3. Mengunduh dataset dari Kaggle

Dataset publik diunduh menggunakan KaggleHub. Pada Google Colab, proses ini biasanya dapat dijalankan langsung tanpa mengunggah file ZIP secara manual.

In [ ]:
# Mengunduh dataset Garbage Classification dari Kaggle.
dataset_path = kagglehub.dataset_download(
    "mostafaabla/garbage-classification"
)

print("Dataset tersimpan di:")
print(dataset_path)

## 4. Menemukan folder utama dataset

Struktur folder hasil unduhan dapat berbeda. Cell ini mencari folder yang berisi subfolder kelas secara otomatis sehingga notebook tidak bergantung pada satu lokasi tertentu.

In [ ]:
# Mencari folder yang langsung berisi seluruh subfolder kelas.
ALL_CLASSES = {
    "battery", "biological", "brown-glass", "cardboard",
    "clothes", "green-glass", "metal", "paper",
    "plastic", "shoes", "trash", "white-glass"
}

candidates = []

for root, dirs, files in os.walk(dataset_path):
    matched_classes = ALL_CLASSES.intersection(set(dirs))
    if len(matched_classes) >= 10:
        candidates.append((root, len(matched_classes)))

if not candidates:
    raise FileNotFoundError(
        "Folder kelas tidak ditemukan. Periksa kembali struktur hasil unduhan dataset."
    )

base_dir = max(candidates, key=lambda item: item[1])[0]

print("Folder utama dataset:")
print(base_dir)
print("\nSubfolder yang ditemukan:")
print(sorted(os.listdir(base_dir)))

## 5. Memeriksa jumlah gambar setiap kelas

Jumlah gambar dihitung langsung dari folder dataset. Pemeriksaan ini untuk melihat ketidakseimbangan kelas sebelum model dilatih.

In [ ]:
# Menghitung jumlah file gambar pada setiap kelas.
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_counts = {}

for class_name in sorted(ALL_CLASSES):
    class_dir = Path(base_dir) / class_name
    if class_dir.exists():
        class_counts[class_name] = sum(
            1
            for file_path in class_dir.rglob("*")
            if file_path.is_file() and file_path.suffix.lower() in VALID_EXTENSIONS
        )

distribution_df = (
    pd.DataFrame(
        [{"kelas": key, "jumlah_gambar": value} for key, value in class_counts.items()]
    )
    .sort_values("jumlah_gambar", ascending=False)
    .reset_index(drop=True)
)

distribution_df

## 6. Menentukan enam kelas yang digunakan

Kelas yang dipilih memiliki distribusi lebih seimbang dibandingkan seluruh dataset. Cell ini juga menghitung rasio kelas terbesar terhadap kelas terkecil.

In [ ]:
# Menentukan kelas inti yang digunakan dalam training.
SELECTED_CLASSES = [
    "battery",
    "biological",
    "cardboard",
    "metal",
    "paper",
    "plastic"
]

selected_distribution = (
    distribution_df[distribution_df["kelas"].isin(SELECTED_CLASSES)]
    .sort_values("kelas")
    .reset_index(drop=True)
)

largest_class = selected_distribution["jumlah_gambar"].max()
smallest_class = selected_distribution["jumlah_gambar"].min()
imbalance_ratio = largest_class / smallest_class

print("Kelas yang digunakan:", SELECTED_CLASSES)
print("Total gambar terpilih:", selected_distribution["jumlah_gambar"].sum())
print(f"Rasio kelas terbesar : terkecil = {imbalance_ratio:.2f} : 1")

selected_distribution

## 7. Menyiapkan data training dan validation

Gambar diubah menjadi ukuran 160 × 160 piksel agar training MobileNetV2 lebih ringan. Data training mendapat augmentasi, sedangkan data validation hanya melalui preprocessing MobileNetV2.

In [ ]:
# Membuat generator data dengan pembagian 80% training dan 20% validation.
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_HEIGHT = 160
IMG_WIDTH = 160
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.20

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=(0.85, 1.15),
    fill_mode="nearest",
    validation_split=VALIDATION_SPLIT
)

validation_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VALIDATION_SPLIT
)

train_generator = train_datagen.flow_from_directory(
    base_dir,
    classes=SELECTED_CLASSES,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED
)

validation_generator = validation_datagen.flow_from_directory(
    base_dir,
    classes=SELECTED_CLASSES,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED
)

## 8. Memeriksa urutan label

Urutan label harus disimpan karena keluaran model berupa indeks. Urutan ini juga digunakan saat menampilkan hasil prediksi.

In [ ]:
# Menampilkan hubungan antara nama kelas dan indeks output model.
class_indices = train_generator.class_indices
index_to_class = {index: name for name, index in class_indices.items()}

INDONESIAN_LABELS = {
    "battery": "Baterai",
    "biological": "Sampah organik",
    "cardboard": "Kardus",
    "metal": "Logam",
    "paper": "Kertas",
    "plastic": "Plastik"
}

print("Pemetaan kelas:")
for index in sorted(index_to_class):
    english_name = index_to_class[index]
    print(f"{index}: {english_name} / {INDONESIAN_LABELS[english_name]}")

## 9. Menampilkan contoh gambar

Satu contoh gambar dari setiap kelas ditampilkan untuk memeriksa apakah folder dan label telah terbaca dengan benar.

In [ ]:
# Menampilkan satu gambar asli dari setiap kelas terpilih.
from tensorflow.keras.preprocessing import image

for class_name in SELECTED_CLASSES:
    class_dir = Path(base_dir) / class_name
    sample_path = next(
        path for path in class_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS
    )

    sample_image = image.load_img(sample_path)
    plt.figure(figsize=(4, 4))
    plt.imshow(sample_image)
    plt.title(f"{class_name} / {INDONESIAN_LABELS[class_name]}")
    plt.axis("off")
    plt.show()

## 10. Membuat model CNN MobileNetV2

MobileNetV2 sudah mempelajari pola visual dari ImageNet. Bagian ekstraksi fitur dibekukan terlebih dahulu, kemudian ditambahkan lapisan klasifikasi untuk enam kelas sampah.

In [ ]:
# Membuat model transfer learning berbasis MobileNetV2.
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

NUM_CLASSES = len(SELECTED_CLASSES)

base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.35),
    layers.Dense(128, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy"
        )
    ]
)

model.summary()

## 11. Menghitung class weight

Walaupun enam kelas yang dipilih relatif seimbang, *class weight* tetap digunakan agar kelas dengan gambar lebih sedikit tidak diabaikan oleh model.

In [ ]:
# Menghitung bobot kelas berdasarkan data training.
train_class_ids = train_generator.classes
unique_class_ids = np.unique(train_class_ids)

weights = compute_class_weight(
    class_weight="balanced",
    classes=unique_class_ids,
    y=train_class_ids
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(unique_class_ids, weights)
}

print("Class weight:")
for class_id, weight in class_weights.items():
    print(
        f"{class_id} - {index_to_class[class_id]}: {weight:.4f}"
    )

## 12. Menyiapkan callback

`EarlyStopping` menghentikan training saat performa tidak membaik, `ReduceLROnPlateau` menurunkan learning rate, dan `ModelCheckpoint` menyimpan model terbaik.

In [ ]:
# Menyiapkan callback untuk menjaga training tetap efisien.
OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

INITIAL_MODEL_PATH = OUTPUT_DIR / "best_initial_garbage_mobilenetv2.keras"
FINE_TUNE_MODEL_PATH = OUTPUT_DIR / "best_finetuned_garbage_mobilenetv2.keras"

def make_callbacks(model_path):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            mode="max",
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.30,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    ]

print("Checkpoint tahap awal:", INITIAL_MODEL_PATH)
print("Checkpoint fine-tuning:", FINE_TUNE_MODEL_PATH)

## 13. Training tahap pertama

Pada tahap ini, bobot MobileNetV2 masih dibekukan. Model hanya melatih lapisan klasifikasi baru sehingga prosesnya lebih cepat dan stabil.

In [ ]:
# Melatih lapisan klasifikasi dengan feature extractor yang dibekukan.
INITIAL_EPOCHS = 8

history_initial = model.fit(
    train_generator,
    epochs=INITIAL_EPOCHS,
    validation_data=validation_generator,
    class_weight=class_weights,
    callbacks=make_callbacks(INITIAL_MODEL_PATH),
    verbose=1
)

## 14. Fine-tuning tahap kedua

Sebagian lapisan akhir MobileNetV2 dibuka agar model menyesuaikan fitur visualnya dengan gambar sampah. Learning rate dibuat sangat kecil agar bobot yang sudah baik tidak rusak.

In [ ]:
# Membuka 30 lapisan terakhir untuk fine-tuning.
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Batch Normalization tetap dibekukan agar fine-tuning lebih stabil.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy"
        )
    ]
)

FINE_TUNE_EPOCHS = 5
initial_epoch_count = len(history_initial.history["loss"])

history_fine = model.fit(
    train_generator,
    initial_epoch=initial_epoch_count,
    epochs=initial_epoch_count + FINE_TUNE_EPOCHS,
    validation_data=validation_generator,
    class_weight=class_weights,
    callbacks=make_callbacks(FINE_TUNE_MODEL_PATH),
    verbose=1
)

# Memilih checkpoint terbaik dari dua tahap training.
best_initial_accuracy = max(history_initial.history["val_accuracy"])
best_fine_accuracy = max(history_fine.history["val_accuracy"])

if best_fine_accuracy >= best_initial_accuracy:
    selected_checkpoint = FINE_TUNE_MODEL_PATH
    selected_stage = "fine-tuning"
else:
    selected_checkpoint = INITIAL_MODEL_PATH
    selected_stage = "training awal"

model = tf.keras.models.load_model(selected_checkpoint)

print(f"Model terbaik berasal dari tahap: {selected_stage}")
print(f"Validation accuracy terbaik: {max(best_initial_accuracy, best_fine_accuracy):.4f}")

## 15. Menggabungkan riwayat training

Riwayat tahap awal dan fine-tuning digabung agar grafik menunjukkan seluruh proses training secara berurutan.

In [ ]:
# Menggabungkan metrik dari dua tahap training.
def combine_history(first_history, second_history):
    combined = {}
    all_keys = set(first_history.history) | set(second_history.history)

    for key in all_keys:
        combined[key] = (
            first_history.history.get(key, [])
            + second_history.history.get(key, [])
        )

    return combined

training_history = combine_history(history_initial, history_fine)

print("Jumlah epoch yang dijalankan:", len(training_history["loss"]))

## 16. Grafik akurasi

Grafik ini membandingkan akurasi training dan validation. Jarak yang terlalu besar dapat menunjukkan overfitting.

In [ ]:
# Menampilkan grafik akurasi pada satu plot.
epochs_range = range(1, len(training_history["accuracy"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_range, training_history["accuracy"], label="Training Accuracy")
plt.plot(epochs_range, training_history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training dan Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 17. Grafik loss

Loss yang menurun pada training dan validation menunjukkan bahwa proses pembelajaran berjalan baik.

In [ ]:
# Menampilkan grafik loss pada plot terpisah.
plt.figure(figsize=(9, 5))
plt.plot(epochs_range, training_history["loss"], label="Training Loss")
plt.plot(epochs_range, training_history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training dan Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## 18. Mengevaluasi model

Cell ini menghitung loss, accuracy, dan top-2 accuracy pada data validation yang tidak diacak.

In [ ]:
# Mengevaluasi performa akhir pada validation set.
validation_generator.reset()

evaluation = model.evaluate(
    validation_generator,
    verbose=1,
    return_dict=True
)

for metric_name, metric_value in evaluation.items():
    print(f"{metric_name}: {metric_value:.4f}")

## 19. Classification report

Classification report menampilkan precision, recall, dan F1-score setiap kelas. Macro average penting karena memberi bobot yang sama kepada seluruh kelas.

In [ ]:
# Membuat classification report untuk seluruh kelas.
validation_generator.reset()

prediction_probabilities = model.predict(
    validation_generator,
    verbose=1
)

predicted_class_ids = np.argmax(
    prediction_probabilities,
    axis=1
)

true_class_ids = validation_generator.classes
class_names = [
    index_to_class[index]
    for index in range(NUM_CLASSES)
]

report = classification_report(
    true_class_ids,
    predicted_class_ids,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose().round(4)
report_df

## 20. Confusion matrix

Confusion matrix menunjukkan pasangan kelas yang paling sering tertukar, misalnya antara kertas, kardus, dan plastik.

In [ ]:
# Menampilkan confusion matrix.
cm = confusion_matrix(
    true_class_ids,
    predicted_class_ids
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 9))
display.plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 21. Menyimpan model dan label

Model disimpan dalam format `.keras`, sedangkan urutan label disimpan dalam file JSON agar dapat digunakan kembali pada aplikasi lain.

In [ ]:
# Menyimpan model final dan label kelas.
FINAL_MODEL_PATH = OUTPUT_DIR / "model_klasifikasi_sampah.keras"
LABEL_PATH = OUTPUT_DIR / "label_klasifikasi_sampah.json"

model.save(FINAL_MODEL_PATH)

with open(LABEL_PATH, "w", encoding="utf-8") as label_file:
    json.dump(
        {
            "class_indices": class_indices,
            "indonesian_labels": INDONESIAN_LABELS,
            "image_size": [IMG_HEIGHT, IMG_WIDTH]
        },
        label_file,
        ensure_ascii=False,
        indent=2
    )

print("Model tersimpan:", FINAL_MODEL_PATH)
print("Label tersimpan:", LABEL_PATH)

## 22. Menguji gambar baru

Unggah satu atau beberapa gambar sampah. Model menampilkan kelas dengan probabilitas tertinggi, confidence score, serta tiga prediksi teratas.

In [ ]:
# Mengunggah gambar baru dan menjalankan prediksi.
from google.colab import files
from tensorflow.keras.preprocessing import image

uploaded = files.upload()

for filename in uploaded.keys():
    uploaded_image = image.load_img(
        filename,
        target_size=(IMG_HEIGHT, IMG_WIDTH)
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(uploaded_image)
    plt.axis("off")
    plt.title(filename)
    plt.show()

    image_array = image.img_to_array(uploaded_image)
    image_array = np.expand_dims(image_array, axis=0)
    image_array = preprocess_input(image_array)

    probabilities = model.predict(
        image_array,
        verbose=0
    )[0]

    top_indices = np.argsort(probabilities)[-3:][::-1]
    best_index = int(top_indices[0])
    best_class = index_to_class[best_index]
    confidence_score = float(probabilities[best_index] * 100)

    print("File:", filename)
    print(
        "Prediksi:",
        f"{best_class} / {INDONESIAN_LABELS[best_class]}"
    )
    print(f"Confidence Score: {confidence_score:.2f}%")
    print("\nTiga prediksi teratas:")

    for rank, class_index in enumerate(top_indices, start=1):
        class_name = index_to_class[int(class_index)]
        score = probabilities[int(class_index)] * 100
        print(
            f"{rank}. {class_name} / "
            f"{INDONESIAN_LABELS[class_name]}: {score:.2f}%"
        )

    print("-" * 50)

## 23. Catatan interpretasi

- Confidence score bukan jaminan bahwa prediksi selalu benar.
- Gunakan gambar yang memperlihatkan satu objek utama dengan cukup jelas.
- Model hanya mengenali enam kelas yang dilatih.
- Gambar pakaian, sepatu, kaca, atau sampah campuran akan tetap dipaksa masuk ke salah satu dari enam kelas.
- Untuk penggunaan nyata, tambahkan kelas `unknown` atau gunakan ambang confidence sebelum menerima prediksi.

## 24. Mengunduh hasil model

Jalankan cell berikut apabila model dan file label ingin diunduh dari Google Colab ke komputer.

In [ ]:
# Mengunduh model dan file label dari Google Colab.
from google.colab import files

files.download(str(FINAL_MODEL_PATH))
files.download(str(LABEL_PATH))